# Week 9: Raster & Remote Sensing

This notebook mirrors the raster analysis you did in **QGIS Week 4** (Raster & Terrain Analysis).

**What you'll learn:**

| Part | Topic | QGIS Equivalent |
|------|-------|------------------|
| A | NDVI & Change Detection | Raster Calculator |
| B | DEM Terrain Analysis | Raster > Analysis > Slope/Hillshade |
| C | Planetary Computer API | (No QGIS equivalent - Python only!) |

---

## Python ↔ QGIS Week 4 Comparison

| QGIS Week 4 Operation | Python Equivalent |
|----------------------|-------------------|
| Raster Calculator (NDVI) | `(nir - red) / (nir + red)` |
| Raster > Analysis > Slope | `np.gradient()` + `np.arctan()` |
| Raster > Analysis > Aspect | `np.arctan2()` |
| Raster > Analysis > Hillshade | Custom function (see Part B) |
| Zonal Statistics | `rasterstats.zonal_stats()` |
| Export as GeoTIFF | `rasterio.open().write()` |

---

## Data options

| Option | Description |
|--------|-------------|
| **Sample data** | Built-in synthetic data. No download needed! |
| **Planetary Computer** | Real satellite data via API (Part C) |
| **Your own data** | Downloaded from Copernicus/USGS |

---

## Step 0: Set up environment

This cell detects whether you're running in Google Colab or local Jupyter.

In [ ]:
# ============================================================
# STEP 0: ENVIRONMENT DETECTION AND PACKAGE INSTALLATION
# ============================================================
# Same pattern as Weeks 7-8. This makes the notebook portable
# between Google Colab and local Jupyter environments.

import sys

# Check if we're in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing packages (takes ~1 minute)...")
    
    # Install geospatial packages
    # rasterio: read/write raster files
    # rasterstats: zonal statistics
    !pip install geopandas rasterio rasterstats -q
    
    print("Done!")
else:
    print("Running in local Jupyter")
    print("Make sure you activated your environment: conda activate intro-gis")

---

## Step 1: Set up folder paths

**QGIS equivalent:** The folder structure you created in Week 1.

In [ ]:
# ============================================================
# STEP 1: SET UP DATA PATHS
# ============================================================

from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    RAW = Path("/content/drive/MyDrive/intro-gis/week09/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week09/data/processed")
else:
    RAW = Path("data/raw")
    PROCESSED = Path("data/processed")

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder: {RAW}")
print(f"Processed folder: {PROCESSED}")

---

## Step 2: Import libraries

**Key libraries for raster analysis:**

| Library | Purpose | QGIS equivalent |
|---------|---------|------------------|
| `numpy` | Array math (rasters are 2D arrays) | Raster Calculator |
| `rasterio` | Read/write raster files | Add Raster Layer, Export |
| `rasterstats` | Zonal statistics | Processing > Zonal Statistics |

In [ ]:
# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

import numpy as np                    # Numerical operations on arrays
import geopandas as gpd               # Vector data
import matplotlib.pyplot as plt       # Plotting
from matplotlib import colors         # Custom colormaps
from shapely.geometry import box      # Create rectangle geometries

import rasterio                       # Read/write raster files
from rasterio.transform import from_bounds  # Create geotransforms

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

---

# Part A: Satellite Imagery & NDVI

**QGIS equivalent:** Raster Calculator in Week 4

NDVI (Normalized Difference Vegetation Index) measures vegetation health
using the ratio of near-infrared to red light reflected by plants.

## Understanding NDVI

### What is NDVI?

**NDVI** exploits a key property of plants:
- Healthy vegetation **absorbs red light** for photosynthesis
- Healthy vegetation **reflects near-infrared (NIR) light**

### The formula (same as QGIS Week 4!)

```
NDVI = (NIR - Red) / (NIR + Red)
```

### Interpreting values

| NDVI | Meaning |
|------|--------|
| 0.6 to 1.0 | Dense, healthy vegetation |
| 0.3 to 0.6 | Moderate vegetation |
| 0.1 to 0.3 | Sparse/stressed vegetation |
| -0.1 to 0.1 | Bare soil, urban areas |
| -1.0 to -0.1 | Water |

In [ ]:
# ============================================================
# STEP 3: LOAD OR GENERATE SATELLITE DATA
# ============================================================
# We check for local files first. If none exist, we generate
# synthetic data that simulates a vegetation clearing event.
#
# QGIS equivalent: Layer > Add Raster Layer

local_before = RAW / "sentinel_before.tif"
local_after = RAW / "sentinel_after.tif"

if local_before.exists() and local_after.exists():
    print("Found local satellite imagery - using your data")
    USE_SAMPLE_DATA = False
else:
    print("No local files found - generating sample data...")
    print("(Add sentinel_before.tif and sentinel_after.tif to data/raw/ to use real imagery)\n")
    USE_SAMPLE_DATA = True
    
    # --------------------------------------------------------
    # GENERATE SYNTHETIC SATELLITE DATA
    # --------------------------------------------------------
    np.random.seed(42)  # For reproducibility
    
    height, width = 100, 100  # 100x100 pixels = 10km x 10km at 100m resolution
    
    # Geographic bounds (Sydney region)
    minx, miny = 151.0, -33.9
    maxx, maxy = 151.1, -33.8
    transform = from_bounds(minx, miny, maxx, maxy, width, height)
    
    # Create "BEFORE" image (healthy vegetation)
    base_ndvi = 0.6 + 0.15 * np.random.randn(height, width)
    
    # Add river (water has negative NDVI)
    river_y = np.sin(np.linspace(0, 2*np.pi, width)) * 10 + 50
    for x in range(width):
        y = int(river_y[x])
        if 0 <= y < height:
            base_ndvi[max(0, y-2):min(height, y+3), x] = -0.2
    
    # Add urban area (low NDVI)
    base_ndvi[0:25, 0:25] = 0.1 + 0.05 * np.random.randn(25, 25)
    
    ndvi_before = np.clip(base_ndvi, -1, 1)
    
    # Create "AFTER" image (with vegetation clearing)
    ndvi_after = ndvi_before.copy()
    ndvi_after[40:70, 50:80] = 0.15 + 0.05 * np.random.randn(30, 30)
    ndvi_after = np.clip(ndvi_after, -1, 1)
    
    # Store for later use
    sample_ndvi = {
        'before': ndvi_before,
        'after': ndvi_after,
        'transform': transform,
        'crs': 'EPSG:4326',
        'bounds': (minx, miny, maxx, maxy)
    }
    raster_transform = transform
    
    # Create analysis zones
    mid_x, mid_y = (minx + maxx) / 2, (miny + maxy) / 2
    zones = gpd.GeoDataFrame({
        'name': ['Northwest', 'Northeast', 'Southwest', 'Southeast'],
        'geometry': [
            box(minx, mid_y, mid_x, maxy),
            box(mid_x, mid_y, maxx, maxy),
            box(minx, miny, mid_x, mid_y),
            box(mid_x, miny, maxx, mid_y)
        ]
    }, crs='EPSG:4326')
    
    print("Sample data generated:")
    print(f"  Image size: {width} × {height} pixels")
    print(f"  Scenario: Forest clearing in southeast quadrant")

In [ ]:
# ============================================================
# STEP 4: VISUALIZE NDVI BEFORE AND AFTER
# ============================================================
# QGIS equivalent: Styling a raster with Singleband Pseudocolor

if USE_SAMPLE_DATA:
    ndvi_before = sample_ndvi['before']
    ndvi_after = sample_ndvi['after']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot BEFORE image
# cmap="RdYlGn" = Red-Yellow-Green (red=low NDVI, green=high)
im1 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI - Before", fontsize=12)
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], shrink=0.8, label="NDVI")

# Plot AFTER image
im2 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI - After", fontsize=12)
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], shrink=0.8, label="NDVI")

plt.suptitle("NDVI: Green = healthy vegetation, Red/Yellow = bare/stressed", y=1.02)
plt.tight_layout()
plt.show()

print("\nNDVI Statistics:")
print(f"Before - Mean: {np.nanmean(ndvi_before):.3f}")
print(f"After  - Mean: {np.nanmean(ndvi_after):.3f}")

In [ ]:
# ============================================================
# STEP 5: CALCULATE AND VISUALIZE NDVI CHANGE
# ============================================================
# Change detection: After - Before
# Positive = vegetation increase, Negative = vegetation loss
#
# QGIS equivalent: Raster Calculator with expression:
# "ndvi_after@1" - "ndvi_before@1"

ndvi_change = ndvi_after - ndvi_before

print("NDVI Change Statistics:")
print(f"  Mean change:  {np.nanmean(ndvi_change):.3f}")
print(f"  Min change:   {np.nanmin(ndvi_change):.3f} (biggest loss)")
print(f"  Max change:   {np.nanmax(ndvi_change):.3f} (biggest gain)")

# Visualize change
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI Before")
axes[0].axis("off")

axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI After")
axes[1].axis("off")

# Change map centered at 0 (red=loss, green=gain)
im = axes[2].imshow(ndvi_change, cmap="RdYlGn", vmin=-0.5, vmax=0.5)
axes[2].set_title("NDVI Change\n(Red = loss, Green = gain)")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], shrink=0.8, label="Change")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 6: ZONAL STATISTICS
# ============================================================
# Summarize raster values within polygon boundaries.
#
# QGIS equivalent: Processing > Toolbox > Zonal Statistics

from rasterstats import zonal_stats

stats = zonal_stats(
    zones,                          # Polygon zones
    ndvi_change,                    # Raster values
    affine=raster_transform,        # Spatial reference
    stats=["mean", "min", "max", "count"],
    nodata=np.nan
)

# Add results to zones GeoDataFrame
zones["ndvi_change_mean"] = [s["mean"] for s in stats]
zones["ndvi_change_min"] = [s["min"] for s in stats]
zones["ndvi_change_max"] = [s["max"] for s in stats]

print("Zonal Statistics - NDVI Change by Zone:")
print(zones[["name", "ndvi_change_mean", "ndvi_change_min", "ndvi_change_max"]].to_string(index=False))

---

# Part B: Digital Elevation Model (DEM) Analysis

**QGIS equivalent:** Raster > Analysis > Slope/Aspect/Hillshade in Week 4

From elevation data, we derive:
- **Slope** — Steepness of terrain
- **Aspect** — Direction the slope faces
- **Hillshade** — Simulated illumination

In [ ]:
# ============================================================
# STEP 7: GENERATE SAMPLE DEM DATA
# ============================================================
# We create synthetic terrain with realistic features.

np.random.seed(123)

dem_height, dem_width = 100, 100

# Create coordinate grids
x = np.linspace(0, 10, dem_width)    # 0-10 km
y = np.linspace(0, 10, dem_height)
X, Y = np.meshgrid(x, y)

# Base terrain (rolling hills using sine waves)
base_terrain = (
    50 * np.sin(X * 0.5) * np.cos(Y * 0.3) +
    20 * np.sin(X * 1.2 + Y * 0.8) +
    10 * np.random.randn(dem_height, dem_width)
)

# Add mountain peak (Gaussian bump)
mountain = 200 * np.exp(-((X - 7)**2 + (Y - 3)**2) / 8)

# Add valley (negative Gaussian)
valley = -80 * np.exp(-((X - 3)**2 + (Y - 7)**2) / 10)

# Combine with 500m base elevation
dem = np.maximum(500 + base_terrain + mountain + valley, 0)

print("Generated sample DEM:")
print(f"  Size: {dem_width} × {dem_height} pixels")
print(f"  Elevation range: {dem.min():.1f}m to {dem.max():.1f}m")

In [ ]:
# ============================================================
# STEP 8: CALCULATE SLOPE
# ============================================================
# Slope = arctan(√(dz/dx)² + (dz/dy)²)
#
# QGIS equivalent: Raster > Analysis > Slope
#
# Key numpy functions:
# - np.gradient(): calculates rate of change between cells
# - np.arctan(): inverse tangent (converts gradient to angle)
# - np.degrees(): converts radians to degrees

cell_size = 100  # meters per pixel

# Calculate gradients (rate of change in x and y directions)
# np.gradient returns [dy, dx] for 2D arrays
dz_dy, dz_dx = np.gradient(dem, cell_size)

# Calculate slope magnitude (Pythagorean theorem)
slope_gradient = np.sqrt(dz_dx**2 + dz_dy**2)

# Convert to degrees
slope_degrees = np.degrees(np.arctan(slope_gradient))

print("Slope calculated!")
print(f"  Min slope:  {slope_degrees.min():.1f}°")
print(f"  Max slope:  {slope_degrees.max():.1f}°")
print(f"  Mean slope: {slope_degrees.mean():.1f}°")
print("\nSlope interpretation:")
print("  0-5°:   Flat (easy walking)")
print("  5-15°:  Moderate")
print("  15-30°: Steep")
print("  >30°:   Very steep")

In [ ]:
# ============================================================
# STEP 9: CALCULATE ASPECT
# ============================================================
# Aspect = compass direction the slope faces
#
# QGIS equivalent: Raster > Analysis > Aspect
#
# Key numpy function:
# - np.arctan2(y, x): returns angle preserving quadrant (-π to π)

# Calculate aspect using arctan2
# Negative gradients because aspect is downhill direction
aspect_radians = np.arctan2(-dz_dx, -dz_dy)

# Convert to degrees
aspect_degrees = np.degrees(aspect_radians)

# Convert from (-180 to 180) to (0 to 360) compass bearing
aspect_degrees = np.where(aspect_degrees < 0, aspect_degrees + 360, aspect_degrees)

print("Aspect calculated!")
print("\nAspect interpretation (compass direction):")
print("  0° / 360°: North-facing")
print("  90°:       East-facing")
print("  180°:      South-facing")
print("  270°:      West-facing")

In [ ]:
# ============================================================
# STEP 10: CALCULATE HILLSHADE
# ============================================================
# Hillshade simulates illumination from a light source.
#
# QGIS equivalent: Raster > Analysis > Hillshade
#
# Parameters:
# - azimuth: light direction (315° = northwest, standard)
# - altitude: sun height (45° = typical)

def calculate_hillshade(elevation, cell_size, azimuth=315, altitude=45):
    """
    Calculate hillshade from a DEM.
    
    Parameters:
    - elevation: 2D array of elevation values
    - cell_size: size of each cell in meters
    - azimuth: light direction in degrees (0=N, 90=E, 180=S, 270=W)
    - altitude: sun height above horizon in degrees
    
    Returns:
    - hillshade: 2D array with values 0 (shadow) to 255 (illuminated)
    """
    # Convert angles to radians
    azimuth_rad = np.radians(360 - azimuth + 90)
    altitude_rad = np.radians(altitude)
    
    # Calculate gradients
    dz_dy, dz_dx = np.gradient(elevation, cell_size)
    
    # Calculate slope and aspect
    slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    aspect = np.arctan2(-dz_dx, dz_dy)
    
    # Hillshade formula (dot product of surface normal and light direction)
    hillshade = (
        np.sin(altitude_rad) * np.cos(slope) +
        np.cos(altitude_rad) * np.sin(slope) * np.cos(azimuth_rad - aspect)
    )
    
    # Scale to 0-255
    return np.clip(hillshade, 0, 1) * 255

hillshade = calculate_hillshade(dem, cell_size=100)

print("Hillshade calculated!")
print(f"  Light direction: 315° (northwest)")
print(f"  Light altitude: 45°")

In [ ]:
# ============================================================
# STEP 11: COMPLETE TERRAIN ANALYSIS VIEW
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Top-left: DEM with hillshade
axes[0, 0].imshow(dem, cmap='terrain')
axes[0, 0].imshow(hillshade, cmap='gray', alpha=0.4)
axes[0, 0].set_title('Elevation with Hillshade', fontsize=12)

# Top-right: Slope
im_slope = axes[0, 1].imshow(slope_degrees, cmap='YlOrRd', vmin=0, vmax=40)
axes[0, 1].set_title('Slope (degrees)', fontsize=12)
plt.colorbar(im_slope, ax=axes[0, 1], shrink=0.8, label='°')

# Bottom-left: Aspect
im_aspect = axes[1, 0].imshow(aspect_degrees, cmap='hsv', vmin=0, vmax=360)
axes[1, 0].set_title('Aspect (compass direction)', fontsize=12)
cbar = plt.colorbar(im_aspect, ax=axes[1, 0], shrink=0.8)
cbar.set_ticks([0, 90, 180, 270, 360])
cbar.set_ticklabels(['N', 'E', 'S', 'W', 'N'])

# Bottom-right: Slope classification
slope_classes = np.zeros_like(slope_degrees)
slope_classes[(slope_degrees >= 0) & (slope_degrees < 5)] = 1
slope_classes[(slope_degrees >= 5) & (slope_degrees < 15)] = 2
slope_classes[(slope_degrees >= 15) & (slope_degrees < 30)] = 3
slope_classes[slope_degrees >= 30] = 4

cmap_classes = colors.ListedColormap(['white', 'green', 'yellow', 'orange', 'red'])
im_class = axes[1, 1].imshow(slope_classes, cmap=cmap_classes, vmin=0, vmax=4)
axes[1, 1].set_title('Slope Classification', fontsize=12)
cbar_class = plt.colorbar(im_class, ax=axes[1, 1], shrink=0.8)
cbar_class.set_ticks([0.5, 1.5, 2.5, 3.5])
cbar_class.set_ticklabels(['Flat\n<5°', 'Moderate\n5-15°', 'Steep\n15-30°', 'Very Steep\n>30°'])

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Complete Terrain Analysis (mirrors QGIS Week 4)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

# Part C: Accessing Real Data via Planetary Computer

**No QGIS equivalent!** This is Python-only functionality.

**Microsoft Planetary Computer** provides free access to petabytes of geospatial data:
- Sentinel-2 satellite imagery
- Landsat imagery
- Copernicus DEM (elevation)
- Land cover datasets

**Why use an API?**
- No large file downloads (100s of MB)
- Access exactly the area and dates you need
- Reproducible workflows

In [ ]:
# ============================================================
# STEP 12: INSTALL PLANETARY COMPUTER PACKAGES
# ============================================================

if IN_COLAB:
    print("Installing Planetary Computer packages...")
    !pip install pystac-client planetary-computer -q
    print("Done!")
else:
    print("Make sure you have installed:")
    print("  pip install pystac-client planetary-computer")

In [ ]:
# ============================================================
# IMPORT PLANETARY COMPUTER LIBRARIES
# ============================================================

try:
    import pystac_client          # Search STAC catalogs
    import planetary_computer     # Microsoft's data catalog
    PC_AVAILABLE = True
    print("Planetary Computer libraries imported!")
    print("\nWhat is STAC?")
    print("STAC = SpatioTemporal Asset Catalog")
    print("It's like a library catalog that helps you find geospatial data.")
except ImportError:
    PC_AVAILABLE = False
    print("Planetary Computer libraries not installed.")
    print("Run the previous cell or: pip install pystac-client planetary-computer")

In [ ]:
# ============================================================
# STEP 13: SEARCH FOR SATELLITE IMAGERY
# ============================================================
# Define your study area and search for Sentinel-2 imagery.
#
# CHANGE THESE COORDINATES TO YOUR AREA OF INTEREST!

if PC_AVAILABLE:
    # Connect to Planetary Computer's STAC catalog
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )
    print("Connected to Planetary Computer!")
    
    # --------------------------------------------------------
    # DEFINE YOUR STUDY AREA
    # --------------------------------------------------------
    # Format: [west_longitude, south_latitude, east_longitude, north_latitude]
    
    # Sydney, Australia (~30km x 22km)
    STUDY_AREA = [151.0, -33.95, 151.3, -33.75]
    
    # Uncomment one of these to try different locations:
    # STUDY_AREA = [144.8, -37.9, 145.1, -37.7]   # Melbourne
    # STUDY_AREA = [-122.5, 37.6, -122.3, 37.8]   # San Francisco
    # STUDY_AREA = [0.0, 51.4, 0.2, 51.6]         # London
    
    print(f"\nStudy area: {STUDY_AREA}")
    
    # --------------------------------------------------------
    # SEARCH FOR SENTINEL-2 IMAGERY
    # --------------------------------------------------------
    search = catalog.search(
        collections=["sentinel-2-l2a"],   # Atmospherically corrected
        bbox=STUDY_AREA,                   # Geographic bounds
        datetime="2024-01-01/2024-12-31",  # Date range
        query={"eo:cloud_cover": {"lt": 10}}  # <10% clouds
    )
    
    items = list(search.items())
    print(f"\nFound {len(items)} Sentinel-2 scenes with <10% cloud cover")
    
    if len(items) > 0:
        print("\nFirst 5 scenes:")
        for item in items[:5]:
            cloud = item.properties.get('eo:cloud_cover', 'N/A')
            print(f"  {item.datetime.strftime('%Y-%m-%d')} - Cloud: {cloud:.1f}%")
else:
    print("Skipping - Planetary Computer not available")

In [ ]:
# ============================================================
# STEP 14: LOAD AND CALCULATE NDVI FROM REAL DATA
# ============================================================

if PC_AVAILABLE and len(items) > 0:
    from rasterio.windows import from_bounds
    
    # Use the most recent scene
    item = items[0]
    print(f"Loading scene: {item.id}")
    print(f"Date: {item.datetime.strftime('%Y-%m-%d')}")
    
    # Get URLs for Red and NIR bands
    red_url = item.assets["B04"].href   # Red band
    nir_url = item.assets["B08"].href   # NIR band
    
    print("\nLoading bands from cloud storage...")
    
    # Load the bands (just the study area window)
    with rasterio.open(red_url) as src:
        window = from_bounds(*STUDY_AREA, src.transform)
        red_band = src.read(1, window=window)
    
    with rasterio.open(nir_url) as src:
        window = from_bounds(*STUDY_AREA, src.transform)
        nir_band = src.read(1, window=window)
    
    print(f"Loaded: {red_band.shape[1]} x {red_band.shape[0]} pixels (10m resolution)")
    
    # Calculate NDVI
    red = np.where(red_band == 0, np.nan, red_band.astype(float))
    nir = np.where(nir_band == 0, np.nan, nir_band.astype(float))
    ndvi_real = np.clip((nir - red) / (nir + red), -1, 1)
    
    print(f"\nNDVI calculated!")
    print(f"  Mean NDVI: {np.nanmean(ndvi_real):.3f}")
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    axes[0].imshow(red_band, cmap='gray')
    axes[0].set_title(f'Red Band (B04)\n{item.datetime.strftime("%Y-%m-%d")}')
    axes[0].axis('off')
    
    axes[1].imshow(nir_band, cmap='gray')
    axes[1].set_title('NIR Band (B08)')
    axes[1].axis('off')
    
    im = axes[2].imshow(ndvi_real, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
    axes[2].set_title('NDVI (calculated)')
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], shrink=0.8, label='NDVI')
    
    plt.suptitle(f'Real Sentinel-2 Data from Planetary Computer', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\nThis is REAL satellite data!")
    print("Change STUDY_AREA coordinates to analyze any location on Earth.")
else:
    print("No data to load. Check previous cells.")

---

## Available Planetary Computer collections

| Collection | Description | Resolution |
|------------|-------------|------------|
| `sentinel-2-l2a` | Sentinel-2 optical imagery | 10-60m |
| `landsat-c2-l2` | Landsat 8/9 imagery | 30m |
| `cop-dem-glo-30` | Copernicus DEM | 30m |
| `io-lulc-9-class` | Land use/land cover | 10m |

**Explore more:** [planetarycomputer.microsoft.com/catalog](https://planetarycomputer.microsoft.com/catalog)

In [ ]:
# ============================================================
# STEP 15: EXPORT RESULTS
# ============================================================

def save_raster(data, path, transform, crs='EPSG:4326'):
    """Save a 2D numpy array as a GeoTIFF."""
    with rasterio.open(
        path, 'w', driver='GTiff',
        height=data.shape[0], width=data.shape[1],
        count=1, dtype=data.dtype,
        crs=crs, transform=transform
    ) as dst:
        dst.write(data, 1)

# Save outputs
zones.to_file(PROCESSED / "ndvi_zones.gpkg", driver="GPKG")
print(f"Saved: {PROCESSED / 'ndvi_zones.gpkg'}")

save_raster(slope_degrees.astype('float32'), PROCESSED / "slope_degrees.tif", raster_transform)
print(f"Saved: {PROCESSED / 'slope_degrees.tif'}")

save_raster(hillshade.astype('float32'), PROCESSED / "hillshade.tif", raster_transform)
print(f"Saved: {PROCESSED / 'hillshade.tif'}")

print("\nAll files saved! You can open these in QGIS.")

---

## Summary: Python ↔ QGIS Week 4

| What you did | Python | QGIS Week 4 |
|--------------|--------|-------------|
| Calculate NDVI | `(nir - red) / (nir + red)` | Raster Calculator |
| Calculate slope | `np.gradient()` + `np.arctan()` | Raster > Analysis > Slope |
| Calculate aspect | `np.arctan2()` | Raster > Analysis > Aspect |
| Calculate hillshade | Custom function | Raster > Analysis > Hillshade |
| Zonal statistics | `rasterstats.zonal_stats()` | Processing > Zonal Statistics |
| Export raster | `rasterio.open().write()` | Export > Save As |
| Access cloud data | Planetary Computer API | (No equivalent!) |

### Key numpy functions

| Function | What it does |
|----------|-------------|
| `np.gradient()` | Rate of change between cells |
| `np.arctan()` | Inverse tangent (for slope) |
| `np.arctan2()` | Inverse tangent preserving quadrant (for aspect) |
| `np.degrees()` | Convert radians to degrees |
| `np.clip()` | Limit values to a range |
| `np.where()` | Conditional selection |

---

## What's next?

**Week 10** covers network analysis in Python:
- Download street networks from OpenStreetMap
- Calculate shortest paths and travel times
- Create service area isochrones

This mirrors what you did in **QGIS Week 6** (Health & Accessibility).

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`